# AI-Based Swimming Health App: End-to-End Colab Prototype

**Author / Student:** Kevin Sun  
**Mentor:** Dr. Qingyang Xiao

**Notebook type:** One-file Google Colab / Jupyter Notebook (`.ipynb`)

Core capabilities:
- smartwatch sensor validation, cleaning, and resampling;
- classical machine-learning stroke classification;
- a deep neural network using engineered motion features;
- a CNN-BiGRU deep sequence model using raw IMU windows;
- a reinforcement-learning coaching policy with safety overrides;
- health and training summaries, plots, a dashboard, and swimmer animation;
- downloadable model, report, data, image, and video artifacts.

> Educational prototype only. It is not a medical device or diagnosis.


In [ ]:
# Environment setup: install only packages that are missing from the current Colab runtime.
# Colab manages its Python interpreter, so this notebook checks the active version rather
# than attempting an unsafe in-place interpreter replacement.

import importlib.util
import subprocess
import sys

PACKAGE_IMPORTS = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "scikit-learn": "sklearn",
    "matplotlib": "matplotlib",
    "plotly": "plotly",
    "torch": "torch",
    "imageio": "imageio",
    "Pillow": "PIL",
    "joblib": "joblib",
}

missing = [pkg for pkg, module in PACKAGE_IMPORTS.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

if sys.version_info < (3, 10):
    raise RuntimeError("Please use a current Colab Python 3 runtime (Python 3.10 or newer).")

LATEST_STABLE_PYTHON_AT_BUILD = "3.14.6"
print("Python runtime:", sys.version.split()[0])
print("Latest stable CPython at notebook build date:", LATEST_STABLE_PYTHON_AT_BUILD)
if sys.version_info[:2] < (3, 14):
    print("Note: Colab controls the interpreter version. This notebook remains compatible with the current managed runtime and Python 3.14.")
print("Environment setup complete.")

In [ ]:
# Imports, reproducibility, configuration, and output folder

from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any
import copy
import json
import math
import os
import random
import shutil
import warnings

import imageio.v2 as imageio
import joblib
import matplotlib.pyplot as plt
from matplotlib.animation import FFMpegWriter, FuncAnimation, PillowWriter
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import HTML, Image, Markdown, Video, display
from scipy import signal
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder, StandardScaler
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore", category=FutureWarning)

@dataclass
class AppConfig:
    random_seed: int = 42
    sample_rate_hz: int = 20
    window_seconds: float = 4.0
    step_seconds: float = 2.0
    minimum_label_purity: float = 0.80
    demo_sessions: int = 24
    feature_dnn_epochs: int = 12
    sequence_model_epochs: int = 7
    batch_size: int = 64
    learning_rate: float = 1e-3
    rl_episodes: int = 3500
    animation_seconds: int = 7
    animation_fps: int = 10
    high_hr_fraction_of_estimated_max: float = 0.95
    spo2_sensor_review_threshold_pct: float = 94.0
    sustained_alert_seconds: int = 10

@dataclass
class AthleteProfile:
    athlete_id: str = "kevin_demo"
    age_years: int = 17  # Editable demonstration value; do not infer a real age from school level.
    mass_kg: float = 65.0
    resting_hr_bpm: float = 60.0
    pool_length_m: float = 25.0

CONFIG = AppConfig()
PROFILE = AthleteProfile()

random.seed(CONFIG.random_seed)
np.random.seed(CONFIG.random_seed)
torch.manual_seed(CONFIG.random_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG.random_seed)
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception:
    pass
try:
    torch.set_num_threads(min(4, os.cpu_count() or 2))
except Exception:
    pass

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = Path("/content/swim_ai_outputs") if Path("/content").exists() else Path.cwd() / "swim_ai_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("Output folder:", OUTPUT_DIR)

## 1. Data ingestion and smartwatch simulator

The simulator is deliberately structured, not random white noise, so this model can learn different stroke rhythms, arm rotations, intensity levels, speed, heart-rate response, rest periods, laps, and sensor noise.

In [ ]:
# Column aliases, validation, file loading, and a structured swimming-watch simulator

COLUMN_ALIASES = {
    "time": "timestamp",
    "datetime": "timestamp",
    "date_time": "timestamp",
    "athlete": "athlete_id",
    "user_id": "athlete_id",
    "session": "session_id",
    "workout_id": "session_id",
    "accelerometer_x": "acc_x",
    "accelerometer_y": "acc_y",
    "accelerometer_z": "acc_z",
    "ax": "acc_x",
    "ay": "acc_y",
    "az": "acc_z",
    "gyroscope_x": "gyro_x",
    "gyroscope_y": "gyro_y",
    "gyroscope_z": "gyro_z",
    "gx": "gyro_x",
    "gy": "gyro_y",
    "gz": "gyro_z",
    "heart_rate": "heart_rate_bpm",
    "hr": "heart_rate_bpm",
    "spo2": "spo2_pct",
    "oxygen_saturation": "spo2_pct",
    "distance": "distance_m",
    "lap": "lap_id",
    "stroke": "stroke_label",
    "style": "stroke_label",
}

MOTION_COLUMNS = ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]
SEQUENCE_CHANNELS = MOTION_COLUMNS + ["acc_mag", "gyro_mag"]
KNOWN_STROKES = ["freestyle", "backstroke", "breaststroke", "butterfly", "rest"]

STROKE_PROFILES = {
    "freestyle": {
        "frequency_hz": 0.78, "speed_mps": 1.25, "intensity": 0.72,
        "acc_amp": (0.42, 0.30, 0.48), "gyro_amp": (115, 80, 165), "phase": (0.0, 1.1, 2.0),
    },
    "backstroke": {
        "frequency_hz": 0.68, "speed_mps": 1.08, "intensity": 0.68,
        "acc_amp": (0.32, 0.46, 0.35), "gyro_amp": (90, 150, 120), "phase": (1.6, 0.3, 2.4),
    },
    "breaststroke": {
        "frequency_hz": 0.50, "speed_mps": 0.88, "intensity": 0.63,
        "acc_amp": (0.58, 0.24, 0.28), "gyro_amp": (75, 95, 65), "phase": (0.2, 2.0, 1.0),
    },
    "butterfly": {
        "frequency_hz": 0.88, "speed_mps": 1.30, "intensity": 0.90,
        "acc_amp": (0.70, 0.44, 0.72), "gyro_amp": (175, 135, 190), "phase": (0.0, 0.6, 1.2),
    },
    "rest": {
        "frequency_hz": 0.10, "speed_mps": 0.0, "intensity": 0.22,
        "acc_amp": (0.04, 0.04, 0.05), "gyro_amp": (8, 8, 8), "phase": (0.0, 0.0, 0.0),
    },
}


def standardize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    renamed = {c: COLUMN_ALIASES.get(str(c).strip().lower(), str(c).strip().lower()) for c in df.columns}
    return df.rename(columns=renamed)


def load_uploaded_watch_file() -> pd.DataFrame:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("The upload widget is available in Google Colab. Use a local CSV/JSON path outside Colab.") from exc

    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No file was uploaded.")
    filename, content = next(iter(uploaded.items()))
    suffix = Path(filename).suffix.lower()
    from io import BytesIO
    buffer = BytesIO(content)
    if suffix == ".csv":
        return pd.read_csv(buffer)
    if suffix in {".json", ".jsonl"}:
        try:
            return pd.read_json(buffer, lines=(suffix == ".jsonl"))
        except ValueError:
            buffer.seek(0)
            return pd.read_json(buffer)
    raise ValueError("Upload a .csv, .json, or .jsonl watch export.")


def generate_synthetic_watch_data(config: AppConfig, n_sessions: int | None = None) -> pd.DataFrame:
    rng = np.random.default_rng(config.random_seed)
    fs = config.sample_rate_hz
    n_sessions = n_sessions or config.demo_sessions
    frames: list[pd.DataFrame] = []

    base_plan = [
        ("freestyle", 18), ("rest", 6),
        ("backstroke", 18), ("rest", 6),
        ("breaststroke", 18), ("rest", 6),
        ("butterfly", 16), ("rest", 8),
    ]

    for session_idx in range(n_sessions):
        athlete_idx = session_idx % 6
        athlete_id = f"athlete_{athlete_idx + 1:02d}"
        session_id = f"session_{session_idx + 1:03d}"
        start_time = pd.Timestamp("2026-01-05 07:00:00") + pd.Timedelta(days=session_idx)
        athlete_factor = rng.normal(1.0, 0.055)
        technique_factor = np.clip(rng.normal(1.0, 0.07), 0.82, 1.17)
        resting_hr = np.clip(56 + athlete_idx * 2 + rng.normal(0, 2), 50, 72)
        estimated_hr_max = 208 - 0.7 * (16 + athlete_idx % 3)
        current_hr = resting_hr + rng.uniform(10, 18)
        elapsed_samples = 0
        cumulative_distance = 0.0
        cumulative_strokes = 0

        # Slightly shuffle stroke order while retaining all five classes in every session.
        swim_segments = base_plan.copy()
        if session_idx % 2:
            swim_segments = [
                ("backstroke", 17), ("rest", 6),
                ("freestyle", 19), ("rest", 6),
                ("butterfly", 15), ("rest", 7),
                ("breaststroke", 19), ("rest", 7),
            ]

        for stroke, duration_s in swim_segments:
            p = STROKE_PROFILES[stroke]
            n = int(duration_s * fs)
            local_t = np.arange(n, dtype=float) / fs
            global_t = (elapsed_samples + np.arange(n)) / fs
            frequency = max(0.05, p["frequency_hz"] * rng.normal(1.0, 0.04) * technique_factor)
            phase0 = rng.uniform(0, 2 * np.pi)
            theta = 2 * np.pi * frequency * local_t + phase0
            second = 2 * theta + 0.35
            fatigue_progress = np.linspace(0, 1, n)
            fatigue_attenuation = 1 - 0.07 * fatigue_progress * (p["intensity"] > 0.7)

            acc_amp = np.asarray(p["acc_amp"]) * athlete_factor * fatigue_attenuation[:, None]
            gyro_amp = np.asarray(p["gyro_amp"]) * athlete_factor * fatigue_attenuation[:, None]
            ph = np.asarray(p["phase"])

            acc_x = acc_amp[:, 0] * np.sin(theta + ph[0]) + 0.12 * acc_amp[:, 0] * np.sin(second)
            acc_y = acc_amp[:, 1] * np.sin(theta + ph[1]) + 0.10 * acc_amp[:, 1] * np.cos(second)
            acc_z = 1.0 + acc_amp[:, 2] * np.sin(theta + ph[2]) + 0.08 * acc_amp[:, 2] * np.sin(second + 0.7)
            gyro_x = gyro_amp[:, 0] * np.sin(theta + ph[0]) + 0.18 * gyro_amp[:, 0] * np.sin(second)
            gyro_y = gyro_amp[:, 1] * np.sin(theta + ph[1]) + 0.15 * gyro_amp[:, 1] * np.cos(second)
            gyro_z = gyro_amp[:, 2] * np.sin(theta + ph[2]) + 0.12 * gyro_amp[:, 2] * np.sin(second + 0.4)

            # Add hardware noise and small swimmer-specific sensor orientation offsets.
            acc_x += rng.normal(0, 0.045, n) + 0.015 * athlete_idx
            acc_y += rng.normal(0, 0.045, n) - 0.010 * athlete_idx
            acc_z += rng.normal(0, 0.055, n)
            gyro_x += rng.normal(0, 5.5, n)
            gyro_y += rng.normal(0, 5.5, n)
            gyro_z += rng.normal(0, 6.5, n)

            if stroke == "rest":
                # Gentle wrist drift while holding the wall.
                acc_x *= 0.45
                acc_y *= 0.45
                acc_z = 1.0 + (acc_z - 1.0) * 0.35
                gyro_x *= 0.35
                gyro_y *= 0.35
                gyro_z *= 0.35

            target_hr = resting_hr + p["intensity"] * (estimated_hr_max - resting_hr) * 0.86
            hr = np.empty(n, dtype=float)
            for i in range(n):
                tau = 28.0 if target_hr > current_hr else 18.0
                current_hr += (target_hr - current_hr) / (tau * fs) + rng.normal(0, 0.10)
                hr[i] = current_hr
            hr += 1.2 * np.sin(2 * np.pi * 0.18 * local_t + rng.uniform(0, 2 * np.pi))
            hr = np.clip(hr, 45, estimated_hr_max + 3)

            spo2 = 98.4 - np.maximum(0, hr - 0.86 * estimated_hr_max) * 0.018
            spo2 += rng.normal(0, 0.18, n)
            spo2 = np.clip(spo2, 92.5, 100.0)

            speed = p["speed_mps"] * athlete_factor * (1 + 0.06 * np.sin(theta))
            speed *= np.clip(1 - 0.05 * fatigue_progress * (p["intensity"] > 0.65), 0.88, 1.05)
            if stroke == "rest":
                speed[:] = 0.0
            segment_distance = np.cumsum(np.maximum(speed, 0)) / fs
            distance = cumulative_distance + segment_distance
            cumulative_distance = float(distance[-1]) if len(distance) else cumulative_distance

            if stroke == "rest":
                stroke_count = np.full(n, cumulative_strokes, dtype=int)
            else:
                stroke_count = cumulative_strokes + np.floor(local_t * frequency).astype(int)
                cumulative_strokes = int(stroke_count[-1])

            timestamps = start_time + pd.to_timedelta(global_t, unit="s")
            frame = pd.DataFrame({
                "timestamp": timestamps,
                "athlete_id": athlete_id,
                "session_id": session_id,
                "acc_x": acc_x,
                "acc_y": acc_y,
                "acc_z": acc_z,
                "gyro_x": gyro_x,
                "gyro_y": gyro_y,
                "gyro_z": gyro_z,
                "heart_rate_bpm": hr,
                "spo2_pct": spo2,
                "water_temp_c": 27.0 + rng.normal(0, 0.08, n),
                "distance_m": distance,
                "lap_id": np.floor(distance / PROFILE.pool_length_m).astype(int) + 1,
                "stroke_count": stroke_count,
                "stroke_label": stroke,
            })
            frames.append(frame)
            elapsed_samples += n

    data = pd.concat(frames, ignore_index=True)

    # Inject a small, realistic amount of missingness for the cleaning pipeline to repair.
    for col in ["acc_x", "gyro_y", "heart_rate_bpm", "spo2_pct"]:
        idx = rng.choice(data.index.to_numpy(), size=max(1, len(data) // 1000), replace=False)
        data.loc[idx, col] = np.nan
    return data

In [ ]:
# Choose the source. Change to "upload" in Colab to select Kevin's CSV/JSON watch export.
DATA_SOURCE = "synthetic"  # @param ["synthetic", "upload"]

if DATA_SOURCE == "upload":
    raw_watch_df = load_uploaded_watch_file()
    source_description = "uploaded watch file"
else:
    raw_watch_df = generate_synthetic_watch_data(CONFIG)
    source_description = "structured synthetic demonstration data"

raw_watch_df = standardize_column_names(raw_watch_df)
print(f"Loaded {len(raw_watch_df):,} samples from {source_description}.")
display(raw_watch_df.head())

## 2. Validation, cleaning, resampling, and data quality

The cleaning stage standardizes names, parses timestamps, removes duplicate samples, interpolate short missing runs, resamples irregular sessions to the configured sampling rate, and derives acceleration and gyroscope magnitudes.

In [ ]:
# Validation, cleaning, resampling, and data-quality diagnostics


def validate_watch_dataframe(df: pd.DataFrame) -> None:
    required = {"timestamp", *MOTION_COLUMNS}
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f"Missing required watch columns: {missing}")


def _sampling_statistics(g: pd.DataFrame) -> tuple[float, float]:
    dt = g["timestamp"].sort_values().diff().dt.total_seconds().dropna()
    if dt.empty or float(dt.median()) <= 0:
        return float("nan"), float("nan")
    hz = 1.0 / float(dt.median())
    jitter_ms = float(dt.std(ddof=0) * 1000) if len(dt) > 1 else 0.0
    return hz, jitter_ms


def resample_session(g: pd.DataFrame, target_hz: int) -> pd.DataFrame:
    g = g.sort_values("timestamp").drop_duplicates("timestamp").copy()
    observed_hz, jitter_ms = _sampling_statistics(g)
    if np.isfinite(observed_hz) and abs(observed_hz - target_hz) / target_hz < 0.03 and jitter_ms < 2.5:
        return g

    period = pd.to_timedelta(1 / target_hz, unit="s")
    indexed = g.set_index("timestamp")
    numeric_cols = indexed.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [c for c in indexed.columns if c not in numeric_cols]

    numeric = indexed[numeric_cols].resample(period).mean().interpolate(method="time").ffill().bfill()
    if categorical_cols:
        categorical = indexed[categorical_cols].resample(period).nearest().ffill().bfill()
        out = pd.concat([numeric, categorical], axis=1)
    else:
        out = numeric
    return out.reset_index()


def clean_watch_data(df: pd.DataFrame, config: AppConfig) -> pd.DataFrame:
    df = standardize_column_names(df.copy())
    validate_watch_dataframe(df)

    if "athlete_id" not in df:
        df["athlete_id"] = PROFILE.athlete_id
    if "session_id" not in df:
        df["session_id"] = "uploaded_session_001"
    if "stroke_label" not in df:
        df["stroke_label"] = "unknown"
    for optional in ["heart_rate_bpm", "spo2_pct", "distance_m", "lap_id", "stroke_count", "water_temp_c"]:
        if optional not in df:
            df[optional] = np.nan

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"]).copy()
    df["athlete_id"] = df["athlete_id"].astype(str)
    df["session_id"] = df["session_id"].astype(str)
    df["stroke_label"] = df["stroke_label"].astype(str).str.lower().str.strip()

    numeric_candidates = list(dict.fromkeys(MOTION_COLUMNS + [
        "heart_rate_bpm", "spo2_pct", "distance_m", "lap_id", "stroke_count", "water_temp_c"
    ]))
    for col in numeric_candidates:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.sort_values(["session_id", "timestamp"]).drop_duplicates(["session_id", "timestamp"])
    interpolated_parts = []
    for _, g in df.groupby("session_id", sort=False):
        g = g.copy()
        g[numeric_candidates] = g[numeric_candidates].interpolate(limit_direction="both", limit=10)
        interpolated_parts.append(resample_session(g, config.sample_rate_hz))
    clean = pd.concat(interpolated_parts, ignore_index=True)

    # Fill residual missing motion values conservatively by session median, then zero as a last resort.
    for col in MOTION_COLUMNS:
        clean[col] = clean.groupby("session_id")[col].transform(lambda s: s.fillna(s.median()))
        clean[col] = clean[col].fillna(0.0)

    clean["acc_mag"] = np.sqrt(clean[["acc_x", "acc_y", "acc_z"]].pow(2).sum(axis=1))
    clean["gyro_mag"] = np.sqrt(clean[["gyro_x", "gyro_y", "gyro_z"]].pow(2).sum(axis=1))
    clean = clean.sort_values(["session_id", "timestamp"]).reset_index(drop=True)
    return clean


def build_data_quality_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for session_id, g in df.groupby("session_id"):
        hz, jitter_ms = _sampling_statistics(g)
        rows.append({
            "session_id": session_id,
            "samples": len(g),
            "duration_min": (g["timestamp"].max() - g["timestamp"].min()).total_seconds() / 60,
            "observed_hz": hz,
            "timing_jitter_ms": jitter_ms,
            "missing_signal_pct": 100 * g[MOTION_COLUMNS].isna().mean().mean(),
            "has_labels": bool((~g["stroke_label"].isin(["unknown", "nan", "none", ""])).any()),
        })
    return pd.DataFrame(rows)


clean_watch_df = clean_watch_data(raw_watch_df, CONFIG)
quality_table = build_data_quality_table(clean_watch_df)

print(f"Clean data: {len(clean_watch_df):,} samples across {clean_watch_df['session_id'].nunique()} sessions.")
display(quality_table.head(10))

## 3. Windowing and motion-feature engineering

Each session is divided into overlapping four-second windows. The deep sequence model receives the raw eight-channel motion tensor. The other models receive statistics, signal energy, percentiles, dominant frequencies, spectral entropy, zero-crossing rates, and corss-axis correlations. Health signals stay outside the stroke-classification feature set to reduce shortcut learning.

In [ ]:
# Fixed-length windows, motion features, and window-level health/training metrics


def dominant_frequency(values: np.ndarray, fs: int, low_hz: float = 0.15, high_hz: float = 2.5) -> float:
    x = np.asarray(values, dtype=float)
    x = np.nan_to_num(x - np.nanmean(x), nan=0.0)
    if len(x) < 4 or np.allclose(x, 0):
        return 0.0
    freqs = np.fft.rfftfreq(len(x), d=1 / fs)
    power = np.abs(np.fft.rfft(x)) ** 2
    mask = (freqs >= low_hz) & (freqs <= high_hz)
    if not mask.any() or np.all(power[mask] <= 0):
        return 0.0
    return float(freqs[mask][np.argmax(power[mask])])


def spectral_entropy(values: np.ndarray) -> float:
    x = np.asarray(values, dtype=float)
    x = np.nan_to_num(x - np.nanmean(x), nan=0.0)
    power = np.abs(np.fft.rfft(x)) ** 2
    power = power[1:]
    total = power.sum()
    if total <= 1e-12 or len(power) <= 1:
        return 0.0
    p = power / total
    return float(-(p * np.log(p + 1e-12)).sum() / np.log(len(p)))


def zero_crossing_rate(values: np.ndarray) -> float:
    x = np.asarray(values, dtype=float)
    x = np.nan_to_num(x - np.nanmean(x), nan=0.0)
    if len(x) < 2:
        return 0.0
    return float(np.mean(np.signbit(x[1:]) != np.signbit(x[:-1])))


def motion_feature_vector(segment: pd.DataFrame, fs: int) -> dict[str, float]:
    features: dict[str, float] = {}
    for col in SEQUENCE_CHANNELS:
        x = segment[col].to_numpy(dtype=float)
        x = np.nan_to_num(x, nan=float(np.nanmedian(x)) if np.isfinite(np.nanmedian(x)) else 0.0)
        centered = x - x.mean()
        features.update({
            f"{col}_mean": float(x.mean()),
            f"{col}_std": float(x.std(ddof=0)),
            f"{col}_min": float(x.min()),
            f"{col}_max": float(x.max()),
            f"{col}_range": float(np.ptp(x)),
            f"{col}_rms": float(np.sqrt(np.mean(x ** 2))),
            f"{col}_q25": float(np.quantile(x, 0.25)),
            f"{col}_q75": float(np.quantile(x, 0.75)),
            f"{col}_zcr": zero_crossing_rate(x),
            f"{col}_entropy": spectral_entropy(x),
        })
        features[f"{col}_dominant_hz"] = dominant_frequency(centered, fs)

    # Cross-axis relationships help distinguish rotational stroke patterns.
    corr = segment[MOTION_COLUMNS].corr().fillna(0.0)
    for a, b in [("acc_x", "acc_y"), ("acc_x", "acc_z"), ("acc_y", "acc_z"),
                 ("gyro_x", "gyro_y"), ("gyro_x", "gyro_z"), ("gyro_y", "gyro_z")]:
        features[f"corr_{a}_{b}"] = float(corr.loc[a, b])
    return features


def safe_mode(values: pd.Series, default: str = "unknown") -> str:
    values = values.dropna().astype(str)
    if values.empty:
        return default
    modes = values.mode()
    return str(modes.iloc[0]) if not modes.empty else default


def build_window_bundle(
    df: pd.DataFrame,
    config: AppConfig,
    require_labels: bool,
) -> dict[str, Any]:
    window_n = int(round(config.window_seconds * config.sample_rate_hz))
    step_n = int(round(config.step_seconds * config.sample_rate_hz))
    feature_rows: list[dict[str, float]] = []
    sequences: list[np.ndarray] = []
    metadata: list[dict[str, Any]] = []

    for session_id, g in df.groupby("session_id", sort=False):
        g = g.sort_values("timestamp").reset_index(drop=True)
        if len(g) < window_n:
            continue
        for start in range(0, len(g) - window_n + 1, step_n):
            segment = g.iloc[start:start + window_n]
            label = safe_mode(segment["stroke_label"])
            label_purity = float((segment["stroke_label"].astype(str) == label).mean())
            label_is_known = label in KNOWN_STROKES
            if require_labels and (not label_is_known or label_purity < config.minimum_label_purity):
                continue

            seq = segment[SEQUENCE_CHANNELS].to_numpy(dtype=np.float32)
            if not np.isfinite(seq).all():
                med = np.nanmedian(seq, axis=0)
                med = np.where(np.isfinite(med), med, 0.0)
                bad = ~np.isfinite(seq)
                seq[bad] = np.take(med, np.where(bad)[1])

            feature_rows.append(motion_feature_vector(segment, config.sample_rate_hz))
            sequences.append(seq)

            duration_s = max((segment["timestamp"].iloc[-1] - segment["timestamp"].iloc[0]).total_seconds(), 1e-6)
            distance = segment["distance_m"].dropna()
            distance_delta = float(distance.iloc[-1] - distance.iloc[0]) if len(distance) >= 2 else float("nan")
            speed_mps = distance_delta / duration_s if np.isfinite(distance_delta) and distance_delta >= 0 else float("nan")
            pace = 100 / speed_mps if np.isfinite(speed_mps) and speed_mps > 0.05 else float("nan")
            gyro_dom = dominant_frequency(segment["gyro_mag"].to_numpy(), config.sample_rate_hz)

            metadata.append({
                "athlete_id": str(segment["athlete_id"].iloc[0]),
                "session_id": str(session_id),
                "window_start": segment["timestamp"].iloc[0],
                "window_end": segment["timestamp"].iloc[-1],
                "true_label": label,
                "label_purity": label_purity,
                "heart_rate_mean_bpm": float(segment["heart_rate_bpm"].mean()),
                "heart_rate_max_bpm": float(segment["heart_rate_bpm"].max()),
                "spo2_min_pct": float(segment["spo2_pct"].min()),
                "distance_delta_m": distance_delta,
                "speed_mps": speed_mps,
                "pace_sec_per_100m": pace,
                "stroke_rate_spm": gyro_dom * 60.0,
                "motion_rms": float(np.sqrt(np.mean(segment["gyro_mag"].to_numpy(dtype=float) ** 2))),
            })

    if not sequences:
        raise ValueError("No valid windows were created. Check session length, timestamps, labels, and sampling rate.")

    features = pd.DataFrame(feature_rows).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return {
        "features": features,
        "sequences": np.stack(sequences).astype(np.float32),
        "meta": pd.DataFrame(metadata),
    }

In [ ]:
# Build the model-training bundle and the bundle that will be analyzed by the AI agent.

recognized_mask = clean_watch_df["stroke_label"].astype(str).str.lower().isin(KNOWN_STROKES)
valid_real_labels = set(clean_watch_df.loc[recognized_mask, "stroke_label"].astype(str).str.lower())
labeled_session_count = clean_watch_df.loc[recognized_mask, "session_id"].astype(str).nunique()
label_session_counts = (
    clean_watch_df.loc[recognized_mask]
    .groupby("stroke_label")["session_id"]
    .nunique()
)
minimum_sessions_per_label = int(label_session_counts.min()) if not label_session_counts.empty else 0
use_uploaded_labels_for_training = (
    len(valid_real_labels) >= 2
    and labeled_session_count >= 6
    and minimum_sessions_per_label >= 3
)

if use_uploaded_labels_for_training:
    training_watch_df = clean_watch_df.copy()
    training_source = "current watch dataset"
else:
    print(
        "The current file does not yet have enough independently labeled sessions for a reliable grouped "
        "train/validation/test split. Creating labeled demo data for model training instead."
    )
    training_watch_df = clean_watch_data(generate_synthetic_watch_data(CONFIG), CONFIG)
    training_source = "synthetic labeled fallback"

training_bundle = build_window_bundle(training_watch_df, CONFIG, require_labels=True)
analysis_bundle = build_window_bundle(clean_watch_df, CONFIG, require_labels=False)

print("Training source:", training_source)
print("Training windows:", len(training_bundle["meta"]))
print("Analysis windows:", len(analysis_bundle["meta"]))
print("Sequence tensor shape [windows, time, channels]:", training_bundle["sequences"].shape)
display(training_bundle["meta"].head())

## 4. Explorarory visualization

The first plot checks whether the session contains sensible motion, physiological resonse, and label timing before training models.

In [ ]:
# Exploratory session plot and basic statistics

SELECTED_SESSION_ID = str(clean_watch_df["session_id"].iloc[0])
selected_raw = clean_watch_df.query("session_id == @SELECTED_SESSION_ID").copy()
selected_raw["elapsed_s"] = (selected_raw["timestamp"] - selected_raw["timestamp"].min()).dt.total_seconds()

stroke_codes = {name: idx for idx, name in enumerate(KNOWN_STROKES)}
colors = plt.cm.tab10(np.linspace(0, 1, len(KNOWN_STROKES)))

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
axes[0].plot(selected_raw["elapsed_s"], selected_raw["heart_rate_bpm"], linewidth=1.5)
axes[0].set_ylabel("Heart rate (bpm)")
axes[0].set_title(f"Session preview: {SELECTED_SESSION_ID}")
axes[0].grid(alpha=0.25)

axes[1].plot(selected_raw["elapsed_s"], selected_raw["acc_mag"], linewidth=1.0, label="Acceleration magnitude")
axes[1].plot(selected_raw["elapsed_s"], selected_raw["gyro_mag"] / 200, linewidth=1.0, label="Gyroscope magnitude / 200")
axes[1].set_ylabel("Scaled motion")
axes[1].legend(loc="upper right")
axes[1].grid(alpha=0.25)

label_numbers = selected_raw["stroke_label"].map(stroke_codes).fillna(-1)
axes[2].scatter(selected_raw["elapsed_s"], label_numbers, c=label_numbers, cmap="tab10", s=5)
axes[2].set_yticks(range(len(KNOWN_STROKES)), KNOWN_STROKES)
axes[2].set_xlabel("Elapsed time (seconds)")
axes[2].set_ylabel("Watch label")
axes[2].grid(alpha=0.25)

fig.tight_layout()
EDA_PATH = OUTPUT_DIR / "session_eda.png"
fig.savefig(EDA_PATH, dpi=150, bbox_inches="tight")
plt.show()

display(selected_raw[["heart_rate_bpm", "spo2_pct", "distance_m", "acc_mag", "gyro_mag"]].describe().round(2))

## 5. Leakage-resistant train, validation, test split and classical machine learning

Sessions, not individual overlapping windows, are separeted into train, validation, and test groups. This prevetns nearly identical neighboring windows from appearing on both sides of evaluation.

In [ ]:
# Grouped split, label encoding, Random Forest training, and reusable evaluation helpers

features_df = training_bundle["features"].copy()
feature_columns = features_df.columns.tolist()
X_features = features_df.to_numpy(dtype=np.float32)
X_sequences = training_bundle["sequences"]
y_text = training_bundle["meta"]["true_label"].astype(str).to_numpy()
groups = training_bundle["meta"]["session_id"].astype(str).to_numpy()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)
class_names = label_encoder.classes_.tolist()
num_classes = len(class_names)

all_class_ids = set(range(num_classes))
for split_attempt in range(60):
    outer = GroupShuffleSplit(
        n_splits=1,
        test_size=0.22,
        random_state=CONFIG.random_seed + split_attempt,
    )
    candidate_train_val, candidate_test = next(outer.split(X_features, y, groups=groups))
    inner = GroupShuffleSplit(
        n_splits=1,
        test_size=0.20,
        random_state=CONFIG.random_seed + 100 + split_attempt,
    )
    candidate_train_rel, candidate_val_rel = next(
        inner.split(
            X_features[candidate_train_val],
            y[candidate_train_val],
            groups=groups[candidate_train_val],
        )
    )
    candidate_train = candidate_train_val[candidate_train_rel]
    candidate_val = candidate_train_val[candidate_val_rel]
    if (
        set(np.unique(y[candidate_train])) == all_class_ids
        and set(np.unique(y[candidate_val])) == all_class_ids
        and set(np.unique(y[candidate_test])) == all_class_ids
    ):
        train_idx, val_idx, test_idx = candidate_train, candidate_val, candidate_test
        break
else:
    raise ValueError(
        "Could not create session-grouped train/validation/test sets containing every class. "
        "Add more independently labeled sessions or use the synthetic fallback."
    )

print("Classes:", class_names)
print(f"Windows — train: {len(train_idx)}, validation: {len(val_idx)}, test: {len(test_idx)}")
print(f"Sessions — train: {len(set(groups[train_idx]))}, validation: {len(set(groups[val_idx]))}, test: {len(set(groups[test_idx]))}")


def metric_summary(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    }


def print_evaluation(name: str, y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    metrics = metric_summary(y_true, y_pred)
    print(f"\n{name}")
    print(json.dumps(metrics, indent=2))
    print(classification_report(
        y_true,
        y_pred,
        labels=np.arange(num_classes),
        target_names=class_names,
        zero_division=0,
    ))
    return metrics


rf_model = RandomForestClassifier(
    n_estimators=260,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=CONFIG.random_seed,
)
rf_model.fit(X_features[train_idx], y[train_idx])
rf_val_prob = rf_model.predict_proba(X_features[val_idx])
rf_test_prob = rf_model.predict_proba(X_features[test_idx])
rf_test_pred = rf_test_prob.argmax(axis=1)
rf_metrics = print_evaluation("Random Forest (classical ML)", y[test_idx], rf_test_pred)

In [ ]:
# Random Forest confusion matrix and feature importance

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
ConfusionMatrixDisplay.from_predictions(
    y[test_idx], rf_test_pred, labels=np.arange(num_classes), display_labels=class_names, normalize="true", cmap="Blues", ax=axes[0], values_format=".2f"
)
axes[0].set_title("Random Forest — normalized confusion matrix")
axes[0].tick_params(axis="x", rotation=35)

importance = pd.Series(rf_model.feature_importances_, index=feature_columns).sort_values(ascending=False).head(16)
importance.sort_values().plot(kind="barh", ax=axes[1])
axes[1].set_title("Top motion features")
axes[1].set_xlabel("Feature importance")
fig.tight_layout()
RF_FIG_PATH = OUTPUT_DIR / "random_forest_evaluation.png"
fig.savefig(RF_FIG_PATH, dpi=150, bbox_inches="tight")
plt.show()

## 6. Deep neural network on egineered features

This multilayer perceptron uses batch normalization, nonlinear layers, dropout, Adam Optimization, and class probabilities. It demonstrates a conventional deep neural network distinct from bot the Random Forest and the raw-sequence model.

In [ ]:
# Shared PyTorch training utilities and the feature DNN


def make_loader(X: np.ndarray, labels: np.ndarray, batch_size: int, shuffle: bool) -> DataLoader:
    dataset = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(labels, dtype=torch.long))
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=0)


def predict_probabilities(model: nn.Module, X: np.ndarray, batch_size: int = 256) -> np.ndarray:
    model.eval()
    loader = DataLoader(TensorDataset(torch.tensor(X, dtype=torch.float32)), batch_size=batch_size, shuffle=False)
    probs = []
    with torch.no_grad():
        for (xb,) in loader:
            logits = model(xb.to(DEVICE))
            probs.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.vstack(probs)


def train_torch_classifier(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int,
    learning_rate: float,
    patience: int = 4,
) -> tuple[nn.Module, pd.DataFrame]:
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.03)
    best_state = copy.deepcopy(model.state_dict())
    best_val = float("inf")
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss_sum = 0.0
        train_correct = 0
        train_count = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            train_loss_sum += float(loss.item()) * len(xb)
            train_correct += int((logits.argmax(1) == yb).sum().item())
            train_count += len(xb)

        model.eval()
        val_loss_sum = 0.0
        val_correct = 0
        val_count = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_loss_sum += float(loss.item()) * len(xb)
                val_correct += int((logits.argmax(1) == yb).sum().item())
                val_count += len(xb)

        train_loss = train_loss_sum / max(train_count, 1)
        val_loss = val_loss_sum / max(val_count, 1)
        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_accuracy": train_correct / max(train_count, 1),
            "val_accuracy": val_correct / max(val_count, 1),
        }
        history.append(row)
        print(
            f"Epoch {epoch:02d}/{epochs} | train loss {train_loss:.4f} | "
            f"val loss {val_loss:.4f} | val accuracy {row['val_accuracy']:.3f}"
        )

        if val_loss < best_val - 1e-4:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print("Early stopping.")
                break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)


class FeatureDNN(nn.Module):
    def __init__(self, n_features: int, n_classes: int):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(n_features, 160),
            nn.BatchNorm1d(160),
            nn.ReLU(),
            nn.Dropout(0.22),
            nn.Linear(160, 96),
            nn.BatchNorm1d(96),
            nn.ReLU(),
            nn.Dropout(0.18),
            nn.Linear(96, 48),
            nn.ReLU(),
            nn.Linear(48, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


feature_scaler = StandardScaler()
X_train_scaled = feature_scaler.fit_transform(X_features[train_idx]).astype(np.float32)
X_val_scaled = feature_scaler.transform(X_features[val_idx]).astype(np.float32)
X_test_scaled = feature_scaler.transform(X_features[test_idx]).astype(np.float32)

feature_dnn = FeatureDNN(X_features.shape[1], num_classes)
feature_dnn, feature_history = train_torch_classifier(
    feature_dnn,
    make_loader(X_train_scaled, y[train_idx], CONFIG.batch_size, True),
    make_loader(X_val_scaled, y[val_idx], CONFIG.batch_size, False),
    epochs=CONFIG.feature_dnn_epochs,
    learning_rate=CONFIG.learning_rate,
)

feature_val_prob = predict_probabilities(feature_dnn, X_val_scaled)
feature_test_prob = predict_probabilities(feature_dnn, X_test_scaled)
feature_test_pred = feature_test_prob.argmax(axis=1)
feature_dnn_metrics = print_evaluation("Feature DNN", y[test_idx], feature_test_pred)

In [ ]:
# Feature DNN learning curves and confusion matrix

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(feature_history["epoch"], feature_history["train_loss"], marker="o", label="train")
axes[0].plot(feature_history["epoch"], feature_history["val_loss"], marker="o", label="validation")
axes[0].set_title("Feature DNN learning curve")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy loss")
axes[0].legend()
axes[0].grid(alpha=0.25)

ConfusionMatrixDisplay.from_predictions(
    y[test_idx], feature_test_pred, labels=np.arange(num_classes), display_labels=class_names, normalize="true", cmap="Purples", ax=axes[1], values_format=".2f"
)
axes[1].set_title("Feature DNN — normalized confusion matrix")
axes[1].tick_params(axis="x", rotation=35)
fig.tight_layout()
FEATURE_DNN_FIG_PATH = OUTPUT_DIR / "feature_dnn_evaluation.png"
fig.savefig(FEATURE_DNN_FIG_PATH, dpi=150, bbox_inches="tight")
plt.show()

## 7. Deep sequence model: 1D CNN + bidirectional GRU

The sequence network constumes the raw motion waveform. Convolutional layers detect local stroke signatures. The directional GRU learns temporal structure across this full window. Global pooling produces the stroke class.

In [ ]:
# Sequence standardization, CNN-BiGRU definition, training, and evaluation

seq_mean = X_sequences[train_idx].mean(axis=(0, 1), keepdims=True)
seq_std = X_sequences[train_idx].std(axis=(0, 1), keepdims=True) + 1e-6
X_seq_normalized = ((X_sequences - seq_mean) / seq_std).astype(np.float32)


class CNNBiGRU(nn.Module):
    def __init__(self, n_channels: int, n_classes: int):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(n_channels, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.12),
        )
        self.gru = nn.GRU(
            input_size=64,
            hidden_size=36,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.head = nn.Sequential(
            nn.Linear(72, 48),
            nn.ReLU(),
            nn.Dropout(0.18),
            nn.Linear(48, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Input shape: [batch, time, channels]
        x = x.transpose(1, 2)
        x = self.conv(x)
        x = x.transpose(1, 2)
        x, _ = self.gru(x)
        x = x.mean(dim=1)
        return self.head(x)


sequence_model = CNNBiGRU(len(SEQUENCE_CHANNELS), num_classes)
sequence_model, sequence_history = train_torch_classifier(
    sequence_model,
    make_loader(X_seq_normalized[train_idx], y[train_idx], CONFIG.batch_size, True),
    make_loader(X_seq_normalized[val_idx], y[val_idx], CONFIG.batch_size, False),
    epochs=CONFIG.sequence_model_epochs,
    learning_rate=CONFIG.learning_rate,
)

sequence_val_prob = predict_probabilities(sequence_model, X_seq_normalized[val_idx])
sequence_test_prob = predict_probabilities(sequence_model, X_seq_normalized[test_idx])
sequence_test_pred = sequence_test_prob.argmax(axis=1)
sequence_metrics = print_evaluation("CNN-BiGRU deep sequence model", y[test_idx], sequence_test_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(sequence_history["epoch"], sequence_history["train_loss"], marker="o", label="train")
axes[0].plot(sequence_history["epoch"], sequence_history["val_loss"], marker="o", label="validation")
axes[0].set_title("Sequence model learning curve")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy loss")
axes[0].legend()
axes[0].grid(alpha=0.25)
ConfusionMatrixDisplay.from_predictions(
    y[test_idx], sequence_test_pred, labels=np.arange(num_classes), display_labels=class_names, normalize="true", cmap="Greens", ax=axes[1], values_format=".2f"
)
axes[1].set_title("CNN-BiGRU — normalized confusion matrix")
axes[1].tick_params(axis="x", rotation=35)
fig.tight_layout()
SEQUENCE_FIG_PATH = OUTPUT_DIR / "sequence_model_evaluation.png"
fig.savefig(SEQUENCE_FIG_PATH, dpi=150, bbox_inches="tight")
plt.show()

## 8. Validation-wegithed ensemble

The agent combines all three models. A model with stronger validation macro-F1 receives more weight, and the test set remains untouched until final evaluation.

In [ ]:
# Validation-weighted probability ensemble

rf_val_f1 = f1_score(y[val_idx], rf_val_prob.argmax(1), average="macro", zero_division=0)
feature_val_f1 = f1_score(y[val_idx], feature_val_prob.argmax(1), average="macro", zero_division=0)
sequence_val_f1 = f1_score(y[val_idx], sequence_val_prob.argmax(1), average="macro", zero_division=0)
raw_weights = np.array([rf_val_f1, feature_val_f1, sequence_val_f1], dtype=float)
ensemble_weights = raw_weights / raw_weights.sum()

ensemble_test_prob = (
    ensemble_weights[0] * rf_test_prob
    + ensemble_weights[1] * feature_test_prob
    + ensemble_weights[2] * sequence_test_prob
)
ensemble_test_pred = ensemble_test_prob.argmax(axis=1)
ensemble_metrics = print_evaluation("Validation-weighted ensemble", y[test_idx], ensemble_test_pred)

print("Ensemble weights:")
for name, weight, score in zip(["Random Forest", "Feature DNN", "CNN-BiGRU"], ensemble_weights, raw_weights):
    print(f"  {name:16s} weight={weight:.3f}, validation macro-F1={score:.3f}")

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y[test_idx], ensemble_test_pred, labels=np.arange(num_classes), display_labels=class_names, normalize="true", cmap="Oranges", ax=ax, values_format=".2f"
)
ax.set_title("Ensemble — normalized confusion matrix")
ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
ENSEMBLE_FIG_PATH = OUTPUT_DIR / "ensemble_confusion_matrix.png"
fig.savefig(ENSEMBLE_FIG_PATH, dpi=150, bbox_inches="tight")
plt.show()

MODEL_METRICS = {
    "random_forest": rf_metrics,
    "feature_dnn": feature_dnn_metrics,
    "cnn_bigru": sequence_metrics,
    "ensemble": ensemble_metrics,
    "ensemble_weights": {
        "random_forest": float(ensemble_weights[0]),
        "feature_dnn": float(ensemble_weights[1]),
        "cnn_bigru": float(ensemble_weights[2]),
    },
}

## 9. Health and training analytics and reinforcement learning

The health layer calculates session duration, distance, pace, heart-rate zones, a demonstration training-load index, oxygen-sensor review flags, technique consistency, fatigue trend, and data quality. The reinforcement-learning environment learns on of four coaching actions:

- recover;
- hold steady
- technique drill
- brief pace increase

A safety layer may override the learned action. The recommendation is for research demonstration only.

In [ ]:
# Health/training report calculations


def estimated_max_hr(age_years: int) -> float:
    # Demonstration estimate only. Individual maximum heart rate can differ substantially.
    return 208.0 - 0.7 * age_years


def heart_rate_zone_number(hr: pd.Series, hr_max_est: float) -> pd.Series:
    fraction = hr / max(hr_max_est, 1.0)
    return pd.cut(
        fraction,
        bins=[-np.inf, 0.60, 0.70, 0.80, 0.90, np.inf],
        labels=[1, 2, 3, 4, 5],
        right=False,
    ).astype(float)


def has_sustained_condition(condition: pd.Series, samples_required: int) -> bool:
    values = condition.fillna(False).astype(int)
    if values.empty:
        return False
    return bool(values.rolling(max(1, samples_required), min_periods=max(1, samples_required)).sum().max() >= samples_required)


def coefficient_of_variation(values: pd.Series) -> float:
    values = pd.to_numeric(values, errors="coerce").dropna()
    values = values[values.abs() > 1e-6]
    if len(values) < 2:
        return 0.0
    return float(values.std(ddof=0) / max(abs(values.mean()), 1e-6))


def technique_consistency_score(window_results: pd.DataFrame) -> float:
    swim = window_results.query("predicted_stroke != 'rest'").copy()
    if swim.empty:
        return 0.0
    per_stroke_scores = []
    for _, g in swim.groupby("predicted_stroke"):
        if len(g) < 2:
            continue
        rate_cv = coefficient_of_variation(g["stroke_rate_spm"])
        motion_cv = coefficient_of_variation(g["motion_rms"])
        score = np.clip(100 - 150 * rate_cv - 80 * motion_cv, 0, 100)
        per_stroke_scores.extend([score] * len(g))
    if not per_stroke_scores:
        return 55.0
    return float(np.mean(per_stroke_scores))


def fatigue_trend_score(session_df: pd.DataFrame, window_results: pd.DataFrame) -> float:
    if len(window_results) < 6:
        return 0.0
    ordered = window_results.sort_values("window_start")
    n = max(2, len(ordered) // 4)
    first = ordered.head(n)
    last = ordered.tail(n)

    hr_first = first["heart_rate_mean_bpm"].mean()
    hr_last = last["heart_rate_mean_bpm"].mean()
    hr_drift = max(0.0, (hr_last - hr_first) / max(abs(hr_first), 1.0)) if np.isfinite(hr_first) else 0.0

    motion_first = first["motion_rms"].mean()
    motion_last = last["motion_rms"].mean()
    motion_drop = max(0.0, (motion_first - motion_last) / max(abs(motion_first), 1.0))

    rate_first = first.query("predicted_stroke != 'rest'")["stroke_rate_spm"].median()
    rate_last = last.query("predicted_stroke != 'rest'")["stroke_rate_spm"].median()
    rate_drift = abs(rate_last - rate_first) / max(abs(rate_first), 1.0) if np.isfinite(rate_first) and np.isfinite(rate_last) else 0.0

    return float(np.clip(420 * hr_drift + 180 * motion_drop + 120 * rate_drift, 0, 100))


def data_quality_score(session_df: pd.DataFrame, target_hz: int) -> float:
    missing = float(session_df[MOTION_COLUMNS].isna().mean().mean())
    observed_hz, jitter_ms = _sampling_statistics(session_df)
    frequency_penalty = 0 if not np.isfinite(observed_hz) else min(1.0, abs(observed_hz - target_hz) / target_hz)
    jitter_penalty = min(1.0, (jitter_ms if np.isfinite(jitter_ms) else 1000) / 25.0)
    return float(np.clip(100 - 100 * missing - 35 * frequency_penalty - 20 * jitter_penalty, 0, 100))


def build_health_report(
    session_df: pd.DataFrame,
    window_results: pd.DataFrame,
    profile: AthleteProfile,
    config: AppConfig,
) -> dict[str, Any]:
    session_df = session_df.sort_values("timestamp").copy()
    duration_s = max((session_df["timestamp"].max() - session_df["timestamp"].min()).total_seconds(), 0.0)
    distance_values = session_df["distance_m"].dropna()
    distance_m = float(distance_values.iloc[-1] - distance_values.iloc[0]) if len(distance_values) >= 2 else float("nan")
    speed_mps = distance_m / duration_s if np.isfinite(distance_m) and duration_s > 0 else float("nan")
    pace_sec_per_100m = 100 / speed_mps if np.isfinite(speed_mps) and speed_mps > 0.05 else float("nan")

    hr = pd.to_numeric(session_df["heart_rate_bpm"], errors="coerce")
    spo2 = pd.to_numeric(session_df["spo2_pct"], errors="coerce")
    hr_max_est = estimated_max_hr(profile.age_years)
    zones = heart_rate_zone_number(hr, hr_max_est)
    dt_minutes = 1 / config.sample_rate_hz / 60
    zone_minutes = {f"zone_{z}_minutes": float((zones == z).sum() * dt_minutes) for z in range(1, 6)}
    training_load_index = float(sum((z ** 2) * (zones == z).sum() * dt_minutes for z in range(1, 6)))

    technique = technique_consistency_score(window_results)
    fatigue = fatigue_trend_score(session_df, window_results)
    quality = data_quality_score(session_df, config.sample_rate_hz)

    stroke_distribution = (
        window_results["predicted_stroke"].value_counts(normalize=True).mul(100).round(2).to_dict()
        if not window_results.empty else {}
    )

    sample_alert_n = int(config.sustained_alert_seconds * config.sample_rate_hz)
    high_hr = has_sustained_condition(
        hr >= config.high_hr_fraction_of_estimated_max * hr_max_est,
        sample_alert_n,
    )
    low_spo2_sensor = has_sustained_condition(
        spo2 < config.spo2_sensor_review_threshold_pct,
        sample_alert_n,
    )

    flags: list[dict[str, str]] = []
    if high_hr:
        flags.append({
            "level": "review",
            "title": "Sustained high-intensity estimate",
            "detail": "The watch heart-rate estimate stayed near the configurable maximum-heart-rate boundary. Slow down or stop if the swimmer feels unwell, and validate this rule clinically before production use.",
        })
    if low_spo2_sensor:
        flags.append({
            "level": "review",
            "title": "Oxygen sensor reading needs review",
            "detail": "A sustained low watch SpO₂ estimate was detected. Check sensor fit and signal quality; seek qualified medical assessment for concerning symptoms. This prototype does not diagnose hypoxemia.",
        })
    if fatigue >= 70:
        flags.append({
            "level": "caution",
            "title": "High fatigue trend",
            "detail": "Late-session heart-rate and motion changes suggest reducing intensity and prioritizing technique or recovery.",
        })
    if quality < 75:
        flags.append({
            "level": "data",
            "title": "Sensor quality limitation",
            "detail": "Missing values, timing irregularity, or unexpected sampling frequency reduced confidence in the analysis.",
        })
    if not flags:
        flags.append({
            "level": "info",
            "title": "No demonstration threshold crossed",
            "detail": "No configured review rule was triggered. This is not a medical clearance or guarantee of safety.",
        })

    median_stroke_rate = window_results.query("predicted_stroke != 'rest'")["stroke_rate_spm"].median()
    report = {
        "project": "Kevin Sun AI Swimming Health App",
        "session_id": str(session_df["session_id"].iloc[0]),
        "athlete_profile": asdict(profile),
        "summary": {
            "duration_minutes": round(duration_s / 60, 2),
            "distance_m": round(distance_m, 2) if np.isfinite(distance_m) else None,
            "average_speed_mps": round(speed_mps, 3) if np.isfinite(speed_mps) else None,
            "pace_sec_per_100m": round(pace_sec_per_100m, 2) if np.isfinite(pace_sec_per_100m) else None,
            "average_heart_rate_bpm": round(float(hr.mean()), 1) if hr.notna().any() else None,
            "maximum_heart_rate_bpm": round(float(hr.max()), 1) if hr.notna().any() else None,
            "estimated_max_heart_rate_bpm": round(hr_max_est, 1),
            "minimum_spo2_sensor_pct": round(float(spo2.min()), 1) if spo2.notna().any() else None,
            "median_estimated_stroke_rate_spm": round(float(median_stroke_rate), 1) if np.isfinite(median_stroke_rate) else None,
            "technique_consistency_score_0_to_100": round(technique, 1),
            "fatigue_trend_score_0_to_100": round(fatigue, 1),
            "data_quality_score_0_to_100": round(quality, 1),
            "demo_training_load_index": round(training_load_index, 2),
        },
        "heart_rate_zones": zone_minutes,
        "predicted_stroke_distribution_pct": stroke_distribution,
        "flags": flags,
        "limitations": [
            "Educational prototype; not a medical device or diagnosis.",
            "Estimated maximum heart rate and alert thresholds are configurable approximations.",
            "Underwater optical heart-rate and SpO₂ readings may be inaccurate.",
            "Synthetic-model performance does not establish accuracy on Kevin's custom watch.",
        ],
    }
    return report

In [ ]:
# Tabular Q-learning environment for personalized swim-coaching actions

RL_ACTIONS = ["recover_easy", "hold_steady", "technique_drill", "brief_pace_increase"]
ACTION_TEXT = {
    "recover_easy": "Use easy swimming or a rest interval; prioritize controlled breathing and stop for concerning symptoms.",
    "hold_steady": "Maintain the current sustainable effort while monitoring technique and comfort.",
    "technique_drill": "Reduce speed slightly and focus on alignment, timing, and consistent stroke rhythm.",
    "brief_pace_increase": "Try a short, controlled pace increase only when comfortable, supervised, and free of warning signs.",
}


class SwimCoachEnv:
    """Small educational MDP: state=(fatigue 0..2, HR zone 0..4, technique 0..2)."""

    def __init__(self, seed: int = 42, horizon: int = 12):
        self.rng = np.random.default_rng(seed)
        self.horizon = horizon
        self.state = (0, 1, 1)
        self.step_count = 0

    def reset(self) -> tuple[int, int, int]:
        self.state = (
            int(self.rng.integers(0, 2)),
            int(self.rng.integers(1, 4)),
            int(self.rng.integers(0, 3)),
        )
        self.step_count = 0
        return self.state

    def step(self, action: int) -> tuple[tuple[int, int, int], float, bool]:
        fatigue, hr_zone, technique = self.state
        noise = int(self.rng.choice([-1, 0, 0, 0, 1]))

        if action == 0:  # recovery
            fatigue -= 1
            hr_zone -= 1
            technique += int(self.rng.random() < 0.25)
        elif action == 1:  # hold
            fatigue += int(self.rng.random() < 0.30)
            hr_zone += noise
        elif action == 2:  # technique drill
            hr_zone -= int(self.rng.random() < 0.55)
            technique += 1
            fatigue -= int(self.rng.random() < 0.25)
        elif action == 3:  # pace increase
            hr_zone += 1
            fatigue += int(self.rng.random() < 0.70)
            technique -= int(fatigue >= 1 and self.rng.random() < 0.55)

        fatigue = int(np.clip(fatigue, 0, 2))
        hr_zone = int(np.clip(hr_zone, 0, 4))
        technique = int(np.clip(technique, 0, 2))
        next_state = (fatigue, hr_zone, technique)

        training_stimulus = [0.0, 0.35, 0.85, 1.0, 0.25][hr_zone]
        technique_reward = 0.55 * technique
        fatigue_penalty = 0.75 * fatigue
        safety_penalty = 4.5 if (hr_zone == 4 and fatigue == 2) else 0.0
        action_context_bonus = 0.0
        if action == 0 and (fatigue == 2 or hr_zone == 4):
            action_context_bonus += 1.5
        if action == 2 and technique == 0:
            action_context_bonus += 1.2
        if action == 3 and fatigue == 0 and hr_zone <= 2 and technique == 2:
            action_context_bonus += 0.7

        reward = training_stimulus + technique_reward - fatigue_penalty - safety_penalty + action_context_bonus
        self.state = next_state
        self.step_count += 1
        return next_state, float(reward), self.step_count >= self.horizon


def train_q_learning(config: AppConfig) -> tuple[np.ndarray, pd.DataFrame]:
    env = SwimCoachEnv(seed=config.random_seed)
    q = np.zeros((3, 5, 3, len(RL_ACTIONS)), dtype=np.float32)
    alpha = 0.14
    gamma = 0.94
    rewards = []

    for episode in range(config.rl_episodes):
        state = env.reset()
        total = 0.0
        epsilon = max(0.04, 0.95 * (1 - episode / config.rl_episodes))
        done = False
        while not done:
            if env.rng.random() < epsilon:
                action = int(env.rng.integers(len(RL_ACTIONS)))
            else:
                action = int(np.argmax(q[state]))
            next_state, reward, done = env.step(action)
            td_target = reward + (0 if done else gamma * float(np.max(q[next_state])))
            q[state + (action,)] += alpha * (td_target - q[state + (action,)])
            state = next_state
            total += reward
        rewards.append(total)
    return q, pd.DataFrame({"episode": np.arange(1, len(rewards) + 1), "reward": rewards})


def report_to_rl_state(report: dict[str, Any]) -> tuple[int, int, int]:
    s = report["summary"]
    fatigue_score = float(s["fatigue_trend_score_0_to_100"])
    fatigue = 0 if fatigue_score < 35 else (1 if fatigue_score < 70 else 2)

    avg_hr = s["average_heart_rate_bpm"]
    hr_max = s["estimated_max_heart_rate_bpm"]
    fraction = (avg_hr / hr_max) if avg_hr is not None and hr_max else 0.65
    hr_zone = int(np.digitize(fraction, [0.60, 0.70, 0.80, 0.90]))
    hr_zone = int(np.clip(hr_zone, 0, 4))

    technique_score = float(s["technique_consistency_score_0_to_100"])
    technique = 2 if technique_score >= 75 else (1 if technique_score >= 50 else 0)
    return fatigue, hr_zone, technique


def constrained_rl_recommendation(q_table: np.ndarray, report: dict[str, Any]) -> dict[str, Any]:
    state = report_to_rl_state(report)
    learned_idx = int(np.argmax(q_table[state]))
    learned_action = RL_ACTIONS[learned_idx]
    final_action = learned_action
    override_reason = None

    flag_titles = {f["title"] for f in report.get("flags", [])}
    fatigue, hr_zone, technique = state
    if hr_zone == 4 or fatigue == 2 or "Sustained high-intensity estimate" in flag_titles:
        final_action = "recover_easy"
        override_reason = "Safety constraint overrode the learned policy because intensity or fatigue was high."
    elif technique == 0:
        final_action = "technique_drill"
        override_reason = "Technique constraint prioritized a drill over increasing effort."

    return {
        "discrete_state": {"fatigue_level_0_to_2": state[0], "hr_zone_0_to_4": state[1], "technique_level_0_to_2": state[2]},
        "learned_action": learned_action,
        "final_safety_constrained_action": final_action,
        "guidance": ACTION_TEXT[final_action],
        "override_reason": override_reason,
        "disclaimer": "Research demonstration only; not medical advice or autonomous supervision.",
    }


q_table, rl_history = train_q_learning(CONFIG)
rl_history["moving_average_reward"] = rl_history["reward"].rolling(120, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(rl_history["episode"], rl_history["moving_average_reward"])
ax.set_title("Q-learning: moving-average episode reward")
ax.set_xlabel("Episode")
ax.set_ylabel("Reward")
ax.grid(alpha=0.25)
fig.tight_layout()
RL_FIG_PATH = OUTPUT_DIR / "reinforcement_learning_curve.png"
fig.savefig(RL_FIG_PATH, dpi=150, bbox_inches="tight")
plt.show()

## 10. AI agent orchestration

Our AI agent runss all three classifiers, fuses their probabilities, calculates health and training metrics, ask the RL policy for an action, applied safey constratins, and returns structured JSON plus a window-level table.

In [ ]:
# End-to-end AI agent


class SwimmingHealthAgent:
    def __init__(
        self,
        rf_model: RandomForestClassifier,
        feature_dnn: nn.Module,
        sequence_model: nn.Module,
        feature_scaler: StandardScaler,
        sequence_mean: np.ndarray,
        sequence_std: np.ndarray,
        label_encoder: LabelEncoder,
        ensemble_weights: np.ndarray,
        feature_columns: list[str],
        q_table: np.ndarray,
        config: AppConfig,
        profile: AthleteProfile,
    ):
        self.rf_model = rf_model
        self.feature_dnn = feature_dnn
        self.sequence_model = sequence_model
        self.feature_scaler = feature_scaler
        self.sequence_mean = sequence_mean
        self.sequence_std = sequence_std
        self.label_encoder = label_encoder
        self.ensemble_weights = ensemble_weights
        self.feature_columns = feature_columns
        self.q_table = q_table
        self.config = config
        self.profile = profile

    def predict_bundle(self, bundle: dict[str, Any]) -> tuple[np.ndarray, np.ndarray]:
        feature_matrix = bundle["features"].reindex(columns=self.feature_columns, fill_value=0.0).to_numpy(dtype=np.float32)
        sequence_matrix = bundle["sequences"].astype(np.float32)

        rf_prob = self.rf_model.predict_proba(feature_matrix)
        scaled_features = self.feature_scaler.transform(feature_matrix).astype(np.float32)
        dnn_prob = predict_probabilities(self.feature_dnn, scaled_features)
        normalized_sequences = ((sequence_matrix - self.sequence_mean) / self.sequence_std).astype(np.float32)
        seq_prob = predict_probabilities(self.sequence_model, normalized_sequences)

        combined = (
            self.ensemble_weights[0] * rf_prob
            + self.ensemble_weights[1] * dnn_prob
            + self.ensemble_weights[2] * seq_prob
        )
        pred = combined.argmax(axis=1)
        return pred, combined

    def run(self, session_id: str, watch_df: pd.DataFrame, bundle: dict[str, Any]) -> tuple[dict[str, Any], pd.DataFrame]:
        meta = bundle["meta"].copy().reset_index(drop=True)
        mask = meta["session_id"].astype(str).eq(str(session_id)).to_numpy()
        if not mask.any():
            raise ValueError(f"No analysis windows found for session {session_id!r}.")

        session_bundle = {
            "features": bundle["features"].loc[mask].reset_index(drop=True),
            "sequences": bundle["sequences"][mask],
            "meta": meta.loc[mask].reset_index(drop=True),
        }
        pred_ids, probabilities = self.predict_bundle(session_bundle)
        results = session_bundle["meta"].copy()
        results["predicted_stroke"] = self.label_encoder.inverse_transform(pred_ids)
        results["prediction_confidence"] = probabilities.max(axis=1)
        for class_idx, class_name in enumerate(self.label_encoder.classes_):
            results[f"prob_{class_name}"] = probabilities[:, class_idx]

        session_df = watch_df[watch_df["session_id"].astype(str).eq(str(session_id))].copy()
        report = build_health_report(session_df, results, self.profile, self.config)
        report["model_information"] = {
            "ensemble_weights": {
                "random_forest": float(self.ensemble_weights[0]),
                "feature_dnn": float(self.ensemble_weights[1]),
                "cnn_bigru": float(self.ensemble_weights[2]),
            },
            "mean_window_confidence": round(float(results["prediction_confidence"].mean()), 4),
            "training_source": training_source,
        }
        report["coaching_recommendation"] = constrained_rl_recommendation(self.q_table, report)
        return report, results


agent = SwimmingHealthAgent(
    rf_model=rf_model,
    feature_dnn=feature_dnn,
    sequence_model=sequence_model,
    feature_scaler=feature_scaler,
    sequence_mean=seq_mean,
    sequence_std=seq_std,
    label_encoder=label_encoder,
    ensemble_weights=ensemble_weights,
    feature_columns=feature_columns,
    q_table=q_table,
    config=CONFIG,
    profile=PROFILE,
)

health_report, window_results = agent.run(SELECTED_SESSION_ID, clean_watch_df, analysis_bundle)

display(Markdown(f"""
### AI agent summary — `{SELECTED_SESSION_ID}`

- **Duration:** {health_report['summary']['duration_minutes']} minutes
- **Distance:** {health_report['summary']['distance_m']} m
- **Average / maximum HR:** {health_report['summary']['average_heart_rate_bpm']} / {health_report['summary']['maximum_heart_rate_bpm']} bpm
- **Technique consistency:** {health_report['summary']['technique_consistency_score_0_to_100']} / 100
- **Fatigue trend:** {health_report['summary']['fatigue_trend_score_0_to_100']} / 100
- **Data quality:** {health_report['summary']['data_quality_score_0_to_100']} / 100
- **Final coaching action:** `{health_report['coaching_recommendation']['final_safety_constrained_action']}`
- **Guidance:** {health_report['coaching_recommendation']['guidance']}
"""))

display(window_results.head(12))
print(json.dumps(health_report, indent=2, default=str))

## 11. Health dashboard and interactive timeline




In [ ]:
# Static dashboard and an interactive Plotly timeline


def create_health_dashboard(
    session_df: pd.DataFrame,
    results: pd.DataFrame,
    report: dict[str, Any],
    output_path: Path,
) -> Path:
    session_df = session_df.sort_values("timestamp").copy()
    session_df["elapsed_min"] = (session_df["timestamp"] - session_df["timestamp"].min()).dt.total_seconds() / 60
    results = results.sort_values("window_start").copy()
    results["elapsed_min"] = (results["window_start"] - session_df["timestamp"].min()).dt.total_seconds() / 60

    fig = plt.figure(figsize=(15, 10))
    grid = fig.add_gridspec(2, 2, height_ratios=[1.1, 1.0])
    ax_hr = fig.add_subplot(grid[0, :])
    ax_strokes = fig.add_subplot(grid[1, 0])
    ax_scores = fig.add_subplot(grid[1, 1])

    ax_hr.plot(session_df["elapsed_min"], session_df["heart_rate_bpm"], linewidth=1.4, label="Watch HR")
    hr_max = report["summary"]["estimated_max_heart_rate_bpm"]
    for fraction, label in [(0.60, "Z2"), (0.70, "Z3"), (0.80, "Z4"), (0.90, "Z5")]:
        ax_hr.axhline(hr_max * fraction, linestyle="--", linewidth=0.9, alpha=0.5, label=label)
    ax_hr.set_title(f"AI Swimming Health Dashboard — {report['session_id']}")
    ax_hr.set_ylabel("Heart rate (bpm)")
    ax_hr.set_xlabel("Elapsed time (minutes)")
    ax_hr.grid(alpha=0.25)
    ax_hr.legend(ncol=5, loc="upper left")

    label_to_y = {label: i for i, label in enumerate(label_encoder.classes_)}
    y_values = results["predicted_stroke"].map(label_to_y)
    scatter = ax_strokes.scatter(
        results["elapsed_min"],
        y_values,
        c=results["prediction_confidence"],
        s=35 + 55 * results["prediction_confidence"],
        cmap="viridis",
        vmin=0,
        vmax=1,
    )
    ax_strokes.set_yticks(range(len(label_to_y)), list(label_to_y.keys()))
    ax_strokes.set_title("Predicted stroke timeline")
    ax_strokes.set_xlabel("Elapsed time (minutes)")
    ax_strokes.grid(alpha=0.25)
    fig.colorbar(scatter, ax=ax_strokes, label="Confidence")

    score_labels = ["Technique", "Fatigue", "Data quality"]
    score_values = [
        report["summary"]["technique_consistency_score_0_to_100"],
        report["summary"]["fatigue_trend_score_0_to_100"],
        report["summary"]["data_quality_score_0_to_100"],
    ]
    bars = ax_scores.barh(score_labels, score_values)
    ax_scores.set_xlim(0, 100)
    ax_scores.set_title("Session scores (fatigue: lower is better)")
    ax_scores.set_xlabel("Score")
    for bar, value in zip(bars, score_values):
        ax_scores.text(min(value + 2, 96), bar.get_y() + bar.get_height() / 2, f"{value:.1f}", va="center")
    recommendation = report["coaching_recommendation"]["final_safety_constrained_action"].replace("_", " ").title()
    ax_scores.text(0.02, -0.28, f"Agent action: {recommendation}", transform=ax_scores.transAxes, fontweight="bold")
    ax_scores.text(0.02, -0.38, "Educational prototype — not medical advice", transform=ax_scores.transAxes, fontsize=9)

    fig.tight_layout()
    fig.savefig(output_path, dpi=170, bbox_inches="tight")
    plt.show()
    return output_path


DASHBOARD_PATH = create_health_dashboard(
    clean_watch_df.query("session_id == @SELECTED_SESSION_ID"),
    window_results,
    health_report,
    OUTPUT_DIR / "swimming_health_dashboard.png",
)

interactive_df = clean_watch_df.query("session_id == @SELECTED_SESSION_ID").copy()
interactive_df["elapsed_s"] = (interactive_df["timestamp"] - interactive_df["timestamp"].min()).dt.total_seconds()
fig_interactive = px.line(
    interactive_df.iloc[::max(1, CONFIG.sample_rate_hz // 4)],
    x="elapsed_s",
    y=["heart_rate_bpm", "spo2_pct"],
    title="Interactive watch health-sensor timeline",
    labels={"value": "Sensor value", "elapsed_s": "Elapsed seconds", "variable": "Signal"},
)
fig_interactive.show()

## 12. Sensor-driven animated swimmer video

The animation is a compact visualization, but not biomechanical reconstruction. Wrist IMU values drive arm and kick angles, while the current predcited stroke and watch heart rate are overlaid. Both GIF and MP4 outputs are generated when mpeg is vailable.

In [ ]:
# Create a 2D swimmer animation driven by watch IMU signals and predicted labels


def create_swimmer_animation(
    session_df: pd.DataFrame,
    results: pd.DataFrame,
    gif_path: Path,
    mp4_path: Path,
    config: AppConfig,
) -> tuple[Path, Path | None]:
    session_df = session_df.sort_values("timestamp").copy()
    max_samples = min(len(session_df), int(config.animation_seconds * config.sample_rate_hz))
    clip = session_df.iloc[:max_samples].copy()
    n_frames = max(2, int(config.animation_seconds * config.animation_fps))
    frame_idx = np.linspace(0, len(clip) - 1, n_frames).astype(int)
    frames = clip.iloc[frame_idx].reset_index(drop=True)

    label_timeline = results[["window_start", "predicted_stroke", "prediction_confidence"]].sort_values("window_start")
    frames = pd.merge_asof(
        frames.sort_values("timestamp"),
        label_timeline,
        left_on="timestamp",
        right_on="window_start",
        direction="nearest",
    )
    frames["predicted_stroke"] = frames["predicted_stroke"].fillna("unknown")
    frames["prediction_confidence"] = frames["prediction_confidence"].fillna(0.0)

    fig, ax = plt.subplots(figsize=(10, 4.6))
    ax.set_xlim(-2.2, 2.2)
    ax.set_ylim(-1.45, 1.35)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title("Watch-driven swimmer animation")
    ax.plot([-2.2, 2.2], [0.75, 0.75], linewidth=2, alpha=0.5)
    for y in [-0.95, -0.65, -0.35, -0.05, 0.25, 0.55]:
        ax.plot([-2.2, 2.2], [y, y + 0.03 * np.sin(y * 7)], linewidth=0.7, alpha=0.18)

    torso, = ax.plot([], [], linewidth=7, solid_capstyle="round")
    head = plt.Circle((0, 0), 0.18, fill=False, linewidth=2)
    watch = plt.Circle((0, 0), 0.07, fill=True)
    ax.add_patch(head)
    ax.add_patch(watch)
    left_arm, = ax.plot([], [], linewidth=4, marker="o", markersize=5)
    right_arm, = ax.plot([], [], linewidth=4, marker="o", markersize=5)
    left_leg, = ax.plot([], [], linewidth=4)
    right_leg, = ax.plot([], [], linewidth=4)
    info = ax.text(-2.05, 1.10, "", fontsize=10, va="top")

    def update(frame_number: int):
        row = frames.iloc[frame_number]
        phase = 2 * np.pi * frame_number / max(n_frames - 1, 1)
        body_bob = 0.06 * np.sin(2 * phase) + 0.0007 * float(row["gyro_y"])
        shoulder = np.array([-0.42, body_bob + 0.10])
        hip = np.array([0.45, body_bob - 0.05])
        neck = np.array([-0.55, body_bob + 0.13])
        head.center = (-0.78, body_bob + 0.17)

        arm_base = np.pi + 0.55 * np.tanh(float(row["gyro_z"]) / 140) + 0.35 * np.sin(phase)
        opposite = np.pi - 0.55 * np.tanh(float(row["gyro_x"]) / 120) - 0.35 * np.sin(phase)
        arm_len = 0.82
        left_hand = shoulder + arm_len * np.array([np.cos(arm_base), np.sin(arm_base)])
        right_hand = shoulder + arm_len * np.array([np.cos(opposite), np.sin(opposite)])

        kick = 0.20 * np.tanh(float(row["acc_y"]) / 0.45) + 0.15 * np.sin(2 * phase)
        foot_x = 1.25
        left_foot = np.array([foot_x, body_bob - 0.12 + kick])
        right_foot = np.array([foot_x, body_bob - 0.12 - kick])

        torso.set_data([neck[0], hip[0]], [neck[1], hip[1]])
        left_arm.set_data([shoulder[0], left_hand[0]], [shoulder[1], left_hand[1]])
        right_arm.set_data([shoulder[0], right_hand[0]], [shoulder[1], right_hand[1]])
        left_leg.set_data([hip[0], left_foot[0]], [hip[1], left_foot[1]])
        right_leg.set_data([hip[0], right_foot[0]], [hip[1], right_foot[1]])
        watch.center = tuple(left_hand)

        hr_value = row.get("heart_rate_bpm", np.nan)
        hr_text = f"{hr_value:.0f} bpm" if np.isfinite(hr_value) else "HR unavailable"
        info.set_text(
            f"Predicted stroke: {row['predicted_stroke']}\n"
            f"Confidence: {row['prediction_confidence']:.0%}\n"
            f"Watch HR: {hr_text}"
        )
        return torso, left_arm, right_arm, left_leg, right_leg, info, head, watch

    animation = FuncAnimation(fig, update, frames=n_frames, interval=1000 / config.animation_fps, blit=False)
    animation.save(gif_path, writer=PillowWriter(fps=config.animation_fps), dpi=110)

    saved_mp4: Path | None = None
    if shutil.which("ffmpeg"):
        try:
            animation.save(mp4_path, writer=FFMpegWriter(fps=config.animation_fps, bitrate=1400), dpi=120)
            saved_mp4 = mp4_path
        except Exception as exc:
            print("MP4 export skipped:", exc)
    plt.close(fig)
    return gif_path, saved_mp4


ANIMATION_GIF_PATH, ANIMATION_MP4_PATH = create_swimmer_animation(
    clean_watch_df.query("session_id == @SELECTED_SESSION_ID"),
    window_results,
    OUTPUT_DIR / "swimmer_motion_animation.gif",
    OUTPUT_DIR / "swimmer_motion_animation.mp4",
    CONFIG,
)

display(Image(filename=str(ANIMATION_GIF_PATH)))
print("GIF:", ANIMATION_GIF_PATH)
print("MP4:", ANIMATION_MP4_PATH if ANIMATION_MP4_PATH else "not available in this runtime")

## 13. Export models, predictions, report, figures, and video

The notebook itself remains the single source file. Running it creates a deployment bundle containing trained model weights, preprocessing parameters, model metrics, session predictions, report JSON, images, animation, and cleaned data.

In [ ]:
# Export artifacts and create one ZIP bundle

REPORT_PATH = OUTPUT_DIR / "health_report.json"
WINDOW_RESULTS_PATH = OUTPUT_DIR / "window_predictions.csv"
CLEAN_DATA_PATH = OUTPUT_DIR / "clean_watch_data.csv"
METRICS_PATH = OUTPUT_DIR / "model_metrics.json"
PREPROCESSING_PATH = OUTPUT_DIR / "preprocessing.json"
RF_MODEL_PATH = OUTPUT_DIR / "random_forest.joblib"
FEATURE_SCALER_PATH = OUTPUT_DIR / "feature_scaler.joblib"
FEATURE_DNN_PATH = OUTPUT_DIR / "feature_dnn_state.pt"
SEQUENCE_MODEL_PATH = OUTPUT_DIR / "cnn_bigru_state.pt"
Q_TABLE_PATH = OUTPUT_DIR / "rl_q_table.npy"

REPORT_PATH.write_text(json.dumps(health_report, indent=2, default=str), encoding="utf-8")
window_results.to_csv(WINDOW_RESULTS_PATH, index=False)
clean_watch_df.to_csv(CLEAN_DATA_PATH, index=False)
METRICS_PATH.write_text(json.dumps(MODEL_METRICS, indent=2), encoding="utf-8")
joblib.dump(rf_model, RF_MODEL_PATH)
joblib.dump(feature_scaler, FEATURE_SCALER_PATH)
torch.save(feature_dnn.state_dict(), FEATURE_DNN_PATH)
torch.save(sequence_model.state_dict(), SEQUENCE_MODEL_PATH)
np.save(Q_TABLE_PATH, q_table)

preprocessing_payload = {
    "config": asdict(CONFIG),
    "profile": asdict(PROFILE),
    "feature_columns": feature_columns,
    "sequence_channels": SEQUENCE_CHANNELS,
    "label_classes": class_names,
    "sequence_mean": seq_mean.reshape(-1).tolist(),
    "sequence_std": seq_std.reshape(-1).tolist(),
    "ensemble_weights": ensemble_weights.tolist(),
    "training_source": training_source,
}
PREPROCESSING_PATH.write_text(json.dumps(preprocessing_payload, indent=2), encoding="utf-8")

ZIP_BASE = OUTPUT_DIR.parent / "kevin_sun_swimming_ai_outputs"
ZIP_PATH = Path(shutil.make_archive(str(ZIP_BASE), "zip", root_dir=OUTPUT_DIR))

artifact_rows = []
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        artifact_rows.append({"artifact": path.name, "size_kb": round(path.stat().st_size / 1024, 1)})
artifact_rows.append({"artifact": ZIP_PATH.name, "size_kb": round(ZIP_PATH.stat().st_size / 1024, 1)})
display(pd.DataFrame(artifact_rows))
print("ZIP bundle:", ZIP_PATH)


def download_from_colab(path: str | Path) -> None:
    """Run download_from_colab(ZIP_PATH) or pass any artifact path in Colab."""
    try:
        from google.colab import files
        files.download(str(path))
    except ImportError:
        print("Not running in Colab. File is available at:", Path(path).resolve())

# Optional in Colab:
# download_from_colab(ZIP_PATH)

In [ ]:
# End-to-end smoke tests

assert len(clean_watch_df) > 0
assert set(MOTION_COLUMNS).issubset(clean_watch_df.columns)
assert np.isfinite(training_bundle["sequences"]).all()
assert len(class_names) >= 2
assert all(0 <= MODEL_METRICS[name]["accuracy"] <= 1 for name in ["random_forest", "feature_dnn", "cnn_bigru", "ensemble"])
assert "coaching_recommendation" in health_report
assert 0 <= health_report["summary"]["data_quality_score_0_to_100"] <= 100
assert DASHBOARD_PATH.exists()
assert ANIMATION_GIF_PATH.exists()
assert ZIP_PATH.exists()

print("All smoke tests passed.")
print("The notebook completed every core path: data → preprocessing → ML → DNN → deep sequence model → RL → agent → report → plots → animation → export.")